<center><ins><h1>Next-Generation Sequencing</h1></ins></center>

<h5>Definition:</h5>
<ul>
    <li>Next-Generation Sequencing or whole genome sequencing (WGS), also known as full genome sequencing, complete genome sequencing, or entire genome sequencing, is the process of determining the entirety, or nearly the entirety, of the DNA sequence of an organism's genome a single time.[2] This entails sequencing all of an organism's chromosomal DNA as well as DNA contained in the mitochondria and, for plants, in the chloroplast.</li>
    <li>No parameter emphasized yet</li>
    <li>Files are imported over .tsv as abundance tables for each sample (pooled from three individual experiments).</li>
</ul>

<center>
<img src="../images/ngs_device.png" alt="Device" width="400" height="300">
<img src="../images/ngs_diagram.png" alt="Diagram" width="489" height="300">
<img src="../images/ngs_example-graph.png" alt="Example-Graph" width="327" height="300">
</center>

###### **Author:** Cedric Hering-Peter
###### **Address:** AG Schulz / Botantical Institut / Am Botanischen Garten 5 / 24118 Kiel

# Packages

In [1]:
# Imports
import os, sys
from scripts import plot_helper
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler 

# Variables

In [2]:
if not data_helper.should_run_ngs(CONFIG):
    raise SystemExit(
        "NGS analysis is disabled in example mode. "
        "Set data_mode: full in analysis_config.yaml to run this notebook."
    )
if not data_helper.should_run_ngs(CONFIG):
    raise SystemExit(
        "NGS analysis is disabled in example mode. "
        "Set data_mode: full in analysis_config.yaml to run this notebook."
    )
TAX_FIL = "s__"
RECAL_SEQS = False
IGNORE_EPS = True

SAMPLES_FULLNAMES = {"CH230921": ["Sewage", "Sewage", "Sewage", "Artificial Medium", "Artificial Medium", "Artificial Medium", "Fish Sewage", "Fish Sewage", "Fish Sewage"],
                 "CH-Physiology": ["0.1 mL (48 h)", "1 mL (48 h)", "10 mL (48 h)", "100 mL (48 h)", "$\\bf{0}$ $\\bf{mL}$ $\\bf{(0 h)}$", "$\\bf{0}$ $\\bf{mL}$ $\\bf{(48 h)}$",  
                 "1500 µmol $m^{\u207B2}$ $s^{\u207B1}$ (48 h)", "375 µmol $m^{\u207B2}$ $s^{\u207B1}$ (48 h)", "750 µmol $m^{\u207B2}$ $s^{\u207B1}$ (48 h)", "1050 µmol $m^{\u207B2}$ $s^{\u207B1}$ (48 h)", "$\\bf{100}$ $\\bf{µmol}$ $\\bf{m^{\u207B2}}$ $\\bf{s^{\u207B1}}$ $\\bf{(0 h)}$", "$\\bf{100}$ $\\bf{µmol}$ $\\bf{m^{\u207B2}}$ $\\bf{s^{\u207B1}}$ $\\bf{(48 h)}$", 
                 "pH 12 (48 h)", "pH 3 (48 h)", "pH 6 (48 h)", "pH 9 (48 h)", "$\\bf{pH}$ $\\bf{7}$ $\\bf{(0 h)}$", "$\\bf{pH}$ $\\bf{7}$ $\\bf{(48 h)}$", 
                 "0.75 PSU (48 h)", "1.5 PSU (48 h)", "15 PSU (48 h)", "35 PSU (48 h)", "$\\bf{0}$ $\\bf{PSU}$ $\\bf{(0 h)}$", "$\\bf{0}$ $\\bf{PSU}$ $\\bf{(48 h)}$", 
                 "15 °C (48 h)", "35 °C (48 h)", "45 °C (48 h)", "5 °C (48 h)", "$\\bf{25}$ $\\bf{°C}$ $\\bf{(0 h)}$", "$\\bf{25}$ $\\bf{°C}$ $\\bf{(48 h)}$"],
                 "CH-Flocculation": ["MBC+$\it{C. vul.}$ (48 h)", "$\\bf{MBC}$ $\\bf{(0 h)}$", "$\\bf{MBC}$ $\\bf{(48 h)}$", 
                 "+EPS (48 h)", "$\\bf{dH_2O}$ $\\bf{(0 h)}$", "$\\bf{dH_2O}$ $\\bf{(48 h)}$", "Antibiotics (48 h)", "$\\bf{MBC}$ $\\bf{(0 h)}$", "$\\bf{MBC}$ $\\bf{(48 h)}$", "Enzymes (48 h)"]}
PHY_LABELS = ["pH 12 (48 h)", "pH 3 (48 h)", "pH 6 (48 h)", "1 mL (48 h)", "35 °C (48 h)", "$\\bf{25}$ $\\bf{°C}$ $\\bf{(0 h)}$", "45 °C (48 h)", "$\\bf{0}$ $\\bf{mL}$ $\\bf{(48 h)}$", "$\\bf{0}$ $\\bf{PSU}$ $\\bf{(48 h)}$"]

# Show numeric output in decimal format e.g., 2.1514
pd.options.display.float_format = '{:,.2f}'.format

# Functions

In [3]:
def normalize_data(sub_df):
    print(f"Shape before transformation: {sub_df.iloc[:, 6:].shape}")
    # Normalizing feature values with StandardScaler
    norm_df = StandardScaler().fit_transform(sub_df.iloc[:, 6:].values)
    print(f"Shape after transformation: {norm_df.shape}")
    # Check if normalized data has a mean of zero and a standard deviation of one
    print(f"Mean: {round(np.mean(norm_df), 1)}\nStandard Deviation: {round(np.std(norm_df))}")
    feat_cols = ["FEATURE_" + str(i+1) for i in range(norm_df.shape[1])]
    norm_df = pd.DataFrame(norm_df, columns = feat_cols)
    return norm_df

def perform_pca(sub_df, n=2):
    pca = PCA(n_components = n)
    PC = pca.fit_transform(sub_df) 
    PC1 = PC[:, 0]
    PC2 = PC[:, 1]
    # Get variance ratios and save in % for pcaplot visualization
    var_PC1 = round(pca.explained_variance_ratio_[0] * 100, 2)
    var_PC2 = round(pca.explained_variance_ratio_[1] * 100, 2)
    print(f"PC1 variance: {var_PC1}\nPC2 variance: {var_PC2}")
    # Create min-max scale for PC1 and PC2
    scale_PC1 = 1.0 / (PC1.max() - PC1.min())
    scale_PC2 = 1.0 / (PC2.max() - PC2.min())
    print(f"PC1 scaling factor: {scale_PC1}\nPC2 scaling factor: {scale_PC2}")
    return PC1, scale_PC1, var_PC1, PC2, scale_PC2, var_PC2

# Raw Import

In [4]:
# Import training data set
files_dir = data_helper.get_instrument_path(CONFIG, "ngs")
raw_df = pd.DataFrame()

for dir_path, dirs, files in os.walk(files_dir):
    if not RECAL_SEQS: break # Save time and import from .csv
    for file in files:
        if file.endswith("taxonomic_profile.tsv"):
            print(f"Working on: {file}")
            ## Extract sample information from file create a data frame
            sam_inf = pd.DataFrame([{"SAMPLE_INFORMATION":os.path.basename(file).split(".taxonomic_profile.tsv")[0]}])
            sam_inf[data_helper.get_sample_columns(META_DATA)] = sam_inf["SAMPLE_INFORMATION"].str.split("_", expand = True)
            sam_inf = sam_inf.drop("SAMPLE_INFORMATION", axis = 1)
            
            # Read taxonomy count data
            file_path = os.path.join(dir_path, file)
            raw_sam = pd.read_csv(file_path, sep = "\t", names = ["Taxonomy", "Count"])
            # Filter for rows that only contain the taxonomy filter the last taxonomy position and append each row to a new data frame 
            fil_sam = pd.DataFrame([(raw_sam.iloc[index].Taxonomy.split("|")[-1], 
                                             raw_sam.iloc[index].Count) for index, entry in raw_sam.iterrows() if TAX_FIL in entry.Taxonomy.split("|")[-1]], 
                                           columns = ["Taxonomy", "Count"])
            # Identify and merge-sum all duplicates
            print(f"{fil_sam.Taxonomy.duplicated().values.sum()} duplicate(s) found and merged in current data frame.")
            clean_sam = fil_sam.groupby("Taxonomy").sum()
            # Merge sample information to the left of (transposed) clean taxonomy data
            clean_sam = pd.concat([sam_inf, clean_sam.T.reset_index()], axis = 1)
            # Append clean data frame to experiment data frame
            raw_df = pd.concat([raw_df, clean_sam], ignore_index = True)

# Custom Cleaning

In [5]:
# Import from .csv to save time
if not RECAL_SEQS:   
    raw_df = pd.read_csv(data_helper.get_output_path(CONFIG, "Raw-Data/Taxa-Variance_Raw-Data.csv")"))

if "index" in raw_df.columns:
    raw_df.drop("index", axis = 1, inplace = True)

# Convert NaN values to zeros and all numbers into integer 
raw_df.iloc[:, 6:] = raw_df.iloc[:, 6:].fillna(0)

# Delete +EPS sample if wished
if IGNORE_EPS:
    raw_df.drop(raw_df[raw_df["SAMPLE_NAME"] == "+EPS"].index, inplace = True)
    SAMPLES_FULLNAMES["CH-Flocculation"] = [sample_appendix for index, sample_appendix in enumerate(SAMPLES_FULLNAMES["CH-Flocculation"]) if index != 3]

print(f"Shape: {raw_df.shape}")
raw_df.sample()

Shape: (49, 17803)


,ID,EXPERIMENT_NAME,SAMPLE_NAME,TIME,BIO_REP,TEC_REP,s__Abiotrophia_defectiva,s__Abyssalbus_ytuae,s__Acanthamoeba_castellanii,s__Acanthoceras_zachariasii,...,s__uncultured_Flexibacteraceae_bacterium,s__uncultured_Methanosaeta_sp,s__uncultured_Minidiscus,s__uncultured_Oocystaceae,s__uncultured_Rhizosolenia,s__uncultured_Selenomonadales_bacterium,s__uncultured_Tribonematales,s__uncultured_Ventricleftida,s__uncultured_bodonid,s__uncultured_marine_cercozoan
22,CH-Physiology,ST,Control,48-h,1,1,2.00,93.00,0.00,2.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


In [6]:
# Custom CHcontrol group with most controls to see annual species drift
CHcontrol = pd.DataFrame(raw_df[(raw_df["SAMPLE_NAME"] == "Control") & (raw_df["TIME"] != "48-h") | 
                                (raw_df["SAMPLE_NAME"] == "OECD") |
                                (raw_df["SAMPLE_NAME"] == "CoA") & (raw_df["TIME"] != "48-h")])

#CHcontrol = pd.DataFrame(CHcontrol[(CHcontrol["EXPERIMENT_NAME"] == "CC") & (CHcontrol["BIO_REP"] == 1)])
CHcontrol.drop(CHcontrol[(CHcontrol["EXPERIMENT_NAME"] == "AT") | (CHcontrol["EXPERIMENT_NAME"] == "PT")].index, inplace = True)
#CHcontrol.drop(CHcontrol[(CHcontrol["EXPERIMENT_NAME"] == "ST") | (CHcontrol["EXPERIMENT_NAME"] == "TT")].index, inplace = True)
#CHcontrol.drop(CHcontrol[(CHcontrol["EXPERIMENT_NAME"] == "CC") & (CHcontrol["BIO_REP"] != 1)].index, inplace = True)


CHcontrol["ID"] = "CHcontrol"
CHcontrol["SAMPLE_NAME"] = ["Summer '24", "Summer '24", "Summer '24", 
                            "Winter '24", "Spring '24", "Winter '24", 
                            "Autumn '24", "Autumn '24", "Autumn '24"]
CHcontrol["TIME"] = "-"

# Normalize current data set
norm_sub_df = normalize_data(CHcontrol)

# Perform PCA forming 2 PCs
PC1, scale_PC1, var_PC1, PC2, scale_PC2, var_PC2 = perform_pca(norm_sub_df, n = 2)

# Scatterplot result
plot_helper.visualize_pcaplot(CHcontrol, "CHcontrol", PHY_LABELS,
                    PC1, scale_PC1, var_PC1, 
                    PC2, scale_PC2, var_PC2,
                    style = None)

Shape before transformation: (9, 17797)
Shape after transformation: (9, 17797)
Mean: 0.0
Standard Deviation: 1
PC1 variance: 27.9
PC2 variance: 20.4
PC1 scaling factor: 0.004909541169220518
PC2 scaling factor: 0.006227549080518132
PCA-Plot "CHcontrol" created and saved.


# File Export

In [7]:
# Export to raw data to csv-file
export_df = pd.DataFrame(raw_df)
file_name = data_helper.get_output_path(CONFIG, "Raw-Data/Taxa-Variance_Raw-Data.csv")
export_df.to_csv(file_name, encoding = "utf-8", index = False)

# Plot Visualization

In [8]:
pca_df = pd.DataFrame(raw_df)

# Delete CH230117 since there is only one sample in it
pca_df.drop(pca_df[pca_df["EXPERIMENT_NAME"] == "SC"].index, inplace = True) 

for id in pca_df["ID"].unique():
    print(f"Experiment group: {id}")
    # Grab subset of data and meta inf
    sub_df = pca_df[pca_df["ID"] == id]
    # Sort and replace sample names with real sample names
    sub_df = sub_df.sort_values(by = ["EXPERIMENT_NAME", "SAMPLE_NAME", "TIME", "BIO_REP"])
    sub_df.loc[:, "SAMPLE_NAME"] = SAMPLES_FULLNAMES.get(id)
    
    # Replace experiment abbrevation with full experiment name for plot asthetics
    plot_df = pd.DataFrame()   
    for exp in sub_df["EXPERIMENT_NAME"].unique():
        exp_sub_df = sub_df[sub_df["EXPERIMENT_NAME"] == exp]
        sub_meta = META_DATA[META_DATA["EXP_ABBR"] == exp][:1].reset_index()
        exp_sub_df.insert(2, "EXP_FULL", sub_meta["EXP_FULL"][0])
        del exp_sub_df["EXPERIMENT_NAME"]
        plot_df = pd.concat([plot_df, exp_sub_df], ignore_index = True)   
    
    # Normalize current data set
    norm_sub_df = normalize_data(plot_df)
    
    # Perform PCA forming 2 PCs
    PC1, scale_PC1, var_PC1, PC2, scale_PC2, var_PC2 = perform_pca(norm_sub_df, n = 2)
    
    # Scatterplot result
    plot_helper.visualize_pcaplot(plot_df, id, PHY_LABELS,
                      PC1, scale_PC1, var_PC1, 
                      PC2, scale_PC2, var_PC2,
                      style = "EXP_FULL"
                      )

Experiment group: CH-Physiology
Shape before transformation: (30, 17797)
Shape after transformation: (30, 17797)
Mean: -0.0
Standard Deviation: 1
PC1 variance: 18.12
PC2 variance: 9.34
PC1 scaling factor: 0.003396074864000755
PC2 scaling factor: 0.006124932921552726
PCA-Plot "CH-Physiology" created and saved.
Experiment group: CH230921
Shape before transformation: (9, 17797)
Shape after transformation: (9, 17797)
Mean: 0.0
Standard Deviation: 1
PC1 variance: 35.74
PC2 variance: 26.75
PC1 scaling factor: 0.004636327394502782
PC2 scaling factor: 0.005930614458582163
PCA-Plot "CH230921" created and saved.
Experiment group: CH-Flocculation
Shape before transformation: (9, 17797)
Shape after transformation: (9, 17797)
Mean: 0.0
Standard Deviation: 1
PC1 variance: 33.32
PC2 variance: 17.84
PC1 scaling factor: 0.005286121091840604
PC2 scaling factor: 0.0063163494103312965
PCA-Plot "CH-Flocculation" created and saved.


# Archive

#### PCA-Analysis (max. components)

In [9]:
# def calc_max_comp(df):
#     if len(df) > 9: return np.arange(9) + 1
#     return np.arange(len(df)) + 1

In [10]:
# Normalizing feature values with StandardScaler
# raw_norm_df = StandardScaler().fit_transform(clean_df.iloc[:, 6:].values)
# print(f"Shape after transformation: {raw_norm_df.shape}")
# # Check if normalized data has a mean of zero and a standard deviation of one
# print(f"Mean: {round(np.mean(raw_norm_df), 1)}\nStandard Deviation: {np.std(raw_norm_df)}")
# # Replace species names with feature since values are more abstract now
# feat_cols = ["FEATURE_" + str(i+1) for i in range(raw_norm_df.shape[1])]
# clean_norm_df = pd.DataFrame(raw_norm_df, columns = feat_cols)
# clean_norm_df.sample(1)

In [11]:
# # Search for ideal number of components (maximum components)
# pca = PCA(n_components = 9)
# pca_max = pca.fit_transform(clean_norm_df)
# print(f"Starting values of maximum PCA: {pca_max[:1]}")
# print(f"Shape: {pca_max.shape}")
# # Extract proportion of explained variance
# prop_var = pca.explained_variance_ratio_ # Alternative scree pcaplot with Kaisers Rule => explained_variance_
# print(f"Proportianl variance array: {prop_var}")
# # Enumerate component numbers
# PC_number = np.arange(pca.n_components_) + 1
# print(f"Principal component number: {PC_number.max()}")

In [12]:
# # Scree pcaplot
# plt.figure(figsize = (4, 2), dpi = 120)
# plt.pcaplot(PC_number,
#          prop_var,
#          "ro-")
# plt.title("Scree pcaplot (Elbow Method)",
#           fontsize = 10)
# plt.xlabel("Component Number",
#            fontsize = 10)
# plt.ylabel("Proportion of Variance",
#            fontsize = 10)
# plt.grid()
# plt.show()

#### Filter options

In [13]:
# # Import single data set
# file_dir = "/Users/cedric/Documents/Career/PhD-Student/Experiments/Instrument-Data/Next-Generation-Sequencing/CH240313/Taxonomic_Profiling/CH-Triplicate_RM_Control_0-h_1_01.taxonomic_profile.tsv"
# # Read taxonomy count data
# raw_sam = pd.read_csv(file_dir, sep = "\t", names = ["Taxonomy", "Count"])
# # Filter for desired domanes
# bac_top = raw_sam[(raw_sam["Taxonomy"].str.contains("d__Bacteria")) & (raw_sam["Taxonomy"].str.contains("g__")) & (raw_sam["Taxonomy"].str.contains("s__") == False)].sort_values(by = "Count", ascending = False)
# cya_top = raw_sam[(raw_sam["Taxonomy"].str.contains("p__Cyanobacteria")) & (raw_sam["Taxonomy"].str.contains("g__")) & (raw_sam["Taxonomy"].str.contains("s__") == False)].sort_values(by = "Count", ascending = False)
# euk_top = raw_sam[(raw_sam["Taxonomy"].str.contains("d__Eukaryota")) & (raw_sam["Taxonomy"].str.contains("g__")) & (raw_sam["Taxonomy"].str.contains("s__") == False)].sort_values(by = "Count", ascending = False)
# chl_top = raw_sam[(raw_sam["Taxonomy"].str.contains("p__Chlorophyta")) & (raw_sam["Taxonomy"].str.contains("g__")) & (raw_sam["Taxonomy"].str.contains("s__") == False)].sort_values(by = "Count", ascending = False)
# fun_top = raw_sam[(raw_sam["Taxonomy"].str.contains("k__Fungi")) & (raw_sam["Taxonomy"].str.contains("g__")) & (raw_sam["Taxonomy"].str.contains("s__") == False)].sort_values(by = "Count", ascending = False)